# Build a Knowledge Graph for LLM Training

End-to-end example: **download a small corpus → build a knowledge graph → store it in a disk-backed graph database (Neo4j) → generate QA training pairs** for fine-tuning an LLM.

Everything uses KGLab's public API, so you can swap in your own documents (JSONL with `id` + `text` fields, see `docs/input_data_format.md`) at any point.

## Motivation for a disk backend

**Real KGs are big.** Production knowledge graphs hold millions of nodes and edges — far too large to fit in RAM. The solution is a **disk-backed** graph store (e.g. **Neo4j**): the graph is streamed straight into the database as it is built, and every query runs as Cypher against the server instead of materialising the whole graph in Python memory. The same pipeline code works unchanged whether the graph lives in RAM, a file, or Neo4j.

**Easy to keep growing.** Because the graph lives in the database, continuing to build on an existing KG is trivial: run the pipeline again with new documents and they are merged into the existing graph (Neo4j upserts by node id, so nothing is duplicated) — the graph grows over time instead of being rebuilt from scratch.


## Prerequisite: Neo4j

This notebook stores the graph in a **disk-backed Neo4j database** instead of RAM.

See **[`docs/neo4j_setup.md`](../docs/neo4j_setup.md)** for how to install and run
Neo4j. The connection credentials are passed directly when creating the pipeline
(cell 3 below), which builds straight into Neo4j.


In [7]:
from kglab.data import Data, RandomSampler
from kglab.pipelines import Baseline

# 1. Fetch a small Wikipedia sample (automatically enriched with hyperlinks)
Data.download(
    "wikipedia",
    path="data/llm_demo/articles.jsonl",
    sampler=RandomSampler(count=5),
)

{'dataset': 'wikipedia',
 'path': 'data/llm_demo/articles.jsonl',
 'cached': True,
 'downloaded': 0}

In [8]:
# 2. Build the knowledge graph and stream it into Neo4j (disk-backed) —
#    nothing is materialised in Python RAM. Zero config otherwise: semantic
#    chunking, layered dedup, spaCy en_core_web_sm.
#    Credentials are loaded from .env (NEO4J_URI / NEO4J_USER / NEO4J_PASSWORD)
#    and passed directly to the pipeline — nothing hardcoded here.
import os

from dotenv import load_dotenv

load_dotenv()  # loads NEO4J_URI / NEO4J_USER / NEO4J_PASSWORD from .env

pipe = Baseline(
    input_paths=["data/llm_demo/"],
    output_dir="output/llm_demo/",
    graph_store_backend="neo4j",  # stream into Neo4j instead of RAM
    graph_store_options={
        "uri": os.environ["NEO4J_URI"],
        "user": os.environ["NEO4J_USER"],
        "password": os.environ["NEO4J_PASSWORD"],
    },
)

kg = pipe.execute()  # preprocess → build → stream to Neo4j in one call

# The graph now lives in Neo4j. The GraphStore read interface is identical
# to the in-RAM version — only each call becomes a live Cypher query.
store = kg["graph_store"]
print(f"KG in Neo4j: {store.number_of_nodes()} nodes, {store.number_of_edges()} edges")
print(
    "Streamed: "
    f"{kg['neo4j_stats']['nodes_written']} nodes, "
    f"{kg['neo4j_stats']['edges_written']} edges"
)

=== Baseline ===
Input:  [PosixPath('data/llm_demo')]
Output: output/llm_demo



Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8216.03it/s]


[preprocess] 20 documents → 139 chunks
[document_relation] 20 doc nodes, 0 doc-doc edges (hyperlink)
[chunks] 139 chunk nodes, 139 chunk→document edges, 119 chunk→next_chunk edges
[entities] 6829 entity→chunk edges
[build_kg] streamed 4265 entities / 13507 triples to Neo4j (4265 nodes, 13507 edges)
[build_kg] 6992 entities → 4265 resolved, 0 nodes, 0 edges
[export] no in-memory graph (stored in the backend) — skipping file export
[manifest] → output/llm_demo/run_manifest.yaml

Done — results in output/llm_demo/
KG in Neo4j: 4265 nodes, 11820 edges
Streamed: 4265 nodes, 13507 edges


## Use the KG for LLM training

The exported knowledge graph (`knowledge_graph.json`) can be used to generate
QA pairs for **supervised fine-tuning (SFT)** — for example, single-hop
questions from entity–relation triples, or multi-hop questions that traverse
the graph.